<a href="https://colab.research.google.com/github/marcocslima/rag_tests/blob/main/Chat_Rag_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y langchain langchain-community langchain-core langchain-google-genai
!pip install -q -U langchain langchain-community langchain-core langchain-google-genai faiss-cpu pypdf gradio
!pip install -q -U langchain-classic
!pip install -q -U sentence-transformers

In [ ]:
import os
from google.colab import drive
from google.colab import userdata

# 1. Montar o Google Drive (se ainda não estiver montado)
drive.mount('/content/drive')

# 2. Puxar a chave do Secrets de forma segura
try:
    minha_chave = userdata.get('API_KEY_MCL')
    # O LangChain exige que a variável de ambiente se chame GOOGLE_API_KEY
    os.environ["GOOGLE_API_KEY"] = minha_chave
    print("✅ Chave API carregada do Secrets com sucesso!")
except userdata.SecretNotFoundError:
    print("❌ ERRO: O secret 'GEMINI_API_KEY' não foi encontrado. Verifique o painel lateral.")

# 3. Definir o caminho do banco FAISS no Drive
DB_PATH = "/content/drive/MyDrive/meu_chat_rag_faiss"
print(f"📂 O banco de dados será salvo em: {DB_PATH}")

# 4. Definir o caminho da pasta com a base de dados à vetorizar
BASE_DATA_PATH = "/content/drive/MyDrive/tmp/RAG/Documentos_Base"

# 5. Definir o MODELO
modelos = ["gemini-2.5-flash-lite","gemini-3-flash-preview"]
MODELO = modelos[0]

In [ ]:
import os
import json
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. NOVO MODELO DE EMBEDDINGS (ABERTO E LOCAL)
# Esse modelo vai baixar cerca de 400MB na primeira vez que rodar
print("Carregando modelo de embeddings local (HuggingFace)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 2. CAMINHO PARA O BANCO
DB_PATH_OPEN = "/content/drive/MyDrive/Projetos/RAG_IA"
if not os.path.exists(DB_PATH_OPEN):
    os.makedirs(DB_PATH_OPEN)

REGISTRY_PATH = os.path.join(DB_PATH_OPEN, "arquivos_processados.json")

def carregar_registro():
    if os.path.exists(REGISTRY_PATH):
        with open(REGISTRY_PATH, 'r') as f:
            return json.load(f)
    return []

def salvar_registro(lista_arquivos):
    with open(REGISTRY_PATH, 'w') as f:
        json.dump(lista_arquivos, f)

def load_or_create_vectorstore():
    if os.path.exists(os.path.join(DB_PATH_OPEN, "index.faiss")):
        print("✅ Banco de dados LOCAL carregado do Drive.")
        return FAISS.load_local(DB_PATH_OPEN, embeddings, allow_dangerous_deserialization=True)
    return None

def add_document_to_db(file_obj):
    global vectorstore
    nome_arquivo = os.path.basename(file_obj.name)
    registro = carregar_registro()

    if nome_arquivo in registro:
        return f"⚠️ O arquivo '{nome_arquivo}' já existe no banco local."

    loader = PyPDFLoader(file_obj.name)
    docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    splits = text_splitter.split_documents(docs)

    print(f"Vetorizando {nome_arquivo} localmente (isso não gasta cota!)...")

    # Como é local, não precisamos de pausas (time.sleep)!
    # Podemos processar tudo de uma vez.
    if vectorstore is None:
        vectorstore = FAISS.from_documents(splits, embeddings)
    else:
        vectorstore.add_documents(splits)

    registro.append(nome_arquivo)
    salvar_registro(registro)
    vectorstore.save_local(DB_PATH_OPEN)

    return f"✅ '{nome_arquivo}' vetorizado localmente com sucesso!"

vectorstore = load_or_create_vectorstore()

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# --- CARGA DO MODELO ---
llm = ChatGoogleGenerativeAI(model=MODELO, temperature=0.1)

# --- 1. PROMPT PARA RECONTEXTUALIZAR PERGUNTAS ---
contextualize_q_system_prompt = (
    "Reescreva a pergunta do usuário para que seja uma pergunta completa e independente, "
    "considerando o histórico do chat. Retorne apenas o texto da pergunta reescrita, "
    "sem aspas e sem explicações."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# --- 2. SEU PROMPT DE CONSULTOR SÊNIOR (Mantido) ---
qa_system_prompt = (
    "Você é um Consultor Sênior e Analista de Inteligência de Dados. Sua missão é fornecer respostas profundas, "
    "analíticas e altamente úteis baseadas estritamente nas premissas e informações do contexto fornecido.\n\n"
    "DIRETRIZES DE COMPORTAMENTO:\n"
    "1. INTELIGÊNCIA ANALÍTICA: Leia nas entrelinhas e correlacione dados.\n"
    "2. CONHECIMENTO EXTERNO: Use apenas como ferramenta instrumental.\n"
    "3. SÍNTESE SEM ALUCINAÇÃO: Não invente fatos.\n\n"
    "Contexto dos Documentos:\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

def get_response(message, history):
    global vectorstore
    if vectorstore is None:
        return "Olá! Por favor, adicione um documento primeiro."

    # Tratamento de input para Gradio 5
    input_text = message.get("text", message.get("content", str(message))) if isinstance(message, dict) else str(message)

    # Formata histórico
    chat_history = []
    for item in history:
        if isinstance(item, (list, tuple)) and len(item) == 2:
            chat_history.append(HumanMessage(content=item[0]))
            chat_history.append(AIMessage(content=item[1]))
        elif isinstance(item, dict):
            r, c = item.get("role"), item.get("content")
            if r == "user": chat_history.append(HumanMessage(content=c))
            elif r == "assistant": chat_history.append(AIMessage(content=c))

    # --- LÓGICA DE RECUPERAÇÃO ---
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    if chat_history:
        # 1. Gera a pergunta reformulada
        # Usamos format_messages para maior compatibilidade
        messages = contextualize_q_prompt.format_messages(input=input_text, chat_history=chat_history)
        rewritten_query_obj = llm.invoke(messages)

        # 2. CORREÇÃO DO ERRO: Trata se o conteúdo for uma lista (formato Gemini 3)
        raw_content = rewritten_query_obj.content
        if isinstance(raw_content, list):
            # Extrai o texto de cada bloco da lista
            search_query = " ".join([str(c.get("text", c)) if isinstance(c, dict) else str(c) for c in raw_content])
        else:
            search_query = str(raw_content)

        search_query = search_query.strip().replace('"', '')
        print(f"🔍 Busca com Memória: {search_query}")
    else:
        search_query = input_text
        print(f"🔍 Busca Direta: {search_query}")

    # --- EXECUÇÃO FINAL ---
    try:
        # Busca documentos
        context_docs = retriever.invoke(search_query)

        # Cria e executa a corrente de resposta
        qa_chain = create_stuff_documents_chain(llm, qa_prompt)
        response = qa_chain.invoke({
            "input": input_text,
            "chat_history": chat_history,
            "context": context_docs
        })

        # Trata se a resposta final também vier como lista
        final_answer = response
        if isinstance(final_answer, list):
            final_answer = " ".join([str(c.get("text", c)) if isinstance(c, dict) else str(c) for c in final_answer])

        return final_answer

    except Exception as e:
        print(f"Erro na execução: {e}")
        return f"Houve um erro ao processar: {str(e)}"

In [ ]:
def vetorizar_pasta_inteira(caminho_da_pasta):
    global vectorstore
    registro = carregar_registro()

    # Lista todos os PDFs da pasta
    arquivos_na_pasta = [f for f in os.listdir(caminho_da_pasta) if f.lower().endswith('.pdf')]

    # Filtra apenas os que ainda NÃO foram processados
    novos_arquivos = [f for f in arquivos_na_pasta if f not in registro]

    if not novos_arquivos:
        print("😎 Todos os arquivos da pasta já estão no banco de dados. Nada a fazer.")
        return

    print(f"🆕 Encontrados {len(novos_arquivos)} novos arquivos para processar.")

    for nome_arq in novos_arquivos:
        caminho_completo = os.path.join(caminho_da_pasta, nome_arq)
        print(f"🚀 Processando: {nome_arq}")

        # Reutilizamos a lógica de upload individual para aproveitar os lotes/pausas
        # Simulamos um objeto de arquivo para a função
        class SimpleFile:
            def __init__(self, name): self.name = name

        resultado = add_document_to_db(SimpleFile(caminho_completo))
        print(resultado)

    print("✅ Processamento da pasta concluído.")

In [ ]:
# Descomente a linha abaixo para rodar a vetorização da pasta:
#vetorizar_pasta_inteira(BASE_DATA_PATH)

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft(), title="Base de Conhecimento") as demo:
    gr.Markdown("# 🤖 Chat RAG")
    gr.Markdown("Converse com seus documentos PDF salvos de forma persistente no seu Google Drive.")

    with gr.Tabs():
        with gr.TabItem("Chat"):

            # --- O SEGREDO ESTÁ AQUI ---
            # Criamos um chatbot personalizado.
            # Você pode usar pixels (ex: height=600, 800) ou viewport height (ex: "70vh" para ocupar 70% da tela)
            chat_visual = gr.Chatbot(height=650, label=MODELO)

            gr.ChatInterface(
                fn=get_response,
                chatbot=chat_visual, # Conectamos o visual maior aqui!
                examples=["Pode me dar um resumo?", "Quais os principais pontos desse documento?", "Faça uma lista com os nomes dos documentos que esta IA tem acesso."],
                cache_examples=False
            )

        with gr.TabItem("Upload de Documentos"):
            gr.Markdown("### 📂 Adicionar Conhecimento")
            file_input = gr.File(label="Selecione um arquivo PDF", file_types=[".pdf"])
            upload_btn = gr.Button("Vetorizar e Salvar no Drive")
            status_txt = gr.Textbox(label="Status do Processamento")

            # O botão de upload manual, caso queira usar em vez da função de ler a pasta
            upload_btn.click(
                fn=add_document_to_db,
                inputs=file_input,
                outputs=status_txt
            )

# Lança a interface com um link público
demo.launch(debug=True, share=True)